## Import Necessary Libraries

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import torch
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from datasets import Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
DATA_PATH = "/kaggle/input/datasets/sharduldhekane/code-mixed-hinglish-hate-speech-detection-dataset/combined_hate_speech_dataset.csv"
DATA_PATH2 = "/kaggle/input/datasets/infiniper/hate-speech-detection-dataset/final_cleaned_dataset.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
sample_df = df.sample(20, random_state=42)[["text", "hate_label", "language"]]
sample_df

In [ ]:
df = df[["text", "hate_label", "language"]].copy()
df.head()

## Basic Cleaning

In [ ]:
def clean_text(text):
    text = str(text)
    
    text = re.sub(r"http\S+", "", text)   # remove urls
    text = re.sub(r"@\w+", "", text)      # mentions
    text = re.sub(r"#\w+", "", text)      # hashtags
    text = re.sub(r"\s+", " ", text)      # extra spaces
    
    return text.strip()

df["text"] = df["text"].apply(clean_text)

In [ ]:
df["text"] = df["text"].apply(clean_text)
df.head()

## Label Distribution


In [ ]:
print(df["hate_label"].value_counts())

## Train-Test-Split

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["hate_label"]
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)

In [ ]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

## Load Tokenizer

In [ ]:
MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## Tokenization

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

In [ ]:
train_dataset = train_dataset.rename_column("hate_label", "labels")
val_dataset = val_dataset.rename_column("hate_label", "labels")

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

## load MuRIL Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

### Metric Function Custom

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions)
    }

### Data Collator- basically dynamic padding

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["hate_label"]),
    y=df["hate_label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

### Training Arguements

In [ ]:
from transformers import Trainer
import torch.nn as nn
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(
            logits.view(-1, 2),   # binary classification
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import EarlyStoppingCallback

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    logging_dir="./logs",
    logging_steps=100,

    report_to="none"
)

### Trainer

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

## Model Evaluation

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print(classification_report(labels, preds))

In [ ]:
print(confusion_matrix(labels, preds))

### Test model

In [ ]:
def predict_text(text):
    trainer.model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = trainer.model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

    print("Probabilities:", probs.cpu().numpy())

    return pred

In [ ]:
texts = [
"You are such a kind and helpful person.",

    "I totally disagree with your opinion but respect your right to say it.",

    "Tum bahut intelligent ho, keep working hard.",

    "This movie was boring and a waste of time.",

    "I hate all people from that religion, they should be removed.",

    "Tum log sab gande ho aur desh barbaad kar rahe ho.",

    "All immigrants are criminals and should be kicked out.",

    "Tu ek ghatiya aadmi hai aur tujhe mar jana chahiye.",

    "Tum chutiya ho."

    "What you said is wrong and insensitive."
]

for t in texts:
    print(t, "->", predict_text(t))

## save model

In [ ]:
trainer.save_model("muril_hate_model")
tokenizer.save_pretrained("muril_hate_model")